# Annotation test run — Kimi K3 on Modal

Runs the same validation/extraction workflow as the Kimi OpenRouter notebook, but calls the project-specific Modal Auto Endpoint directly:

`https://utsavtandon96--ep-kimi-k3-server.us-west.modal.direct`

The endpoint serves `moonshotai/Kimi-K3`. Model-specific request and authentication logic lives in `extension/scripts/model_specific/modal_kimi.py`; shared prompt construction, validation, caching, and scoring remain unchanged.

Results use their own cache namespace, `modal-endpoint/kimi-k3-{effort}`, so they cannot overwrite the OpenRouter or Moonshot-direct Kimi results. Re-runs skip valid cache records and retry invalid ones.

**Authentication:** the script reads `MODAL-KEY` and `MODAL-SECRET` from the repo-root `.env` file (or `MODAL_KEY` and `MODAL_SECRET` from the process environment). Credentials are never printed or written to cache.

**Cost accounting:** Modal Auto Endpoints are billed by compute time and do not return a per-request dollar cost. The cache retains token counts and latency, but `cost_usd = 0.0` means *unaccounted*, not free.

In [1]:
import os, sys
from pathlib import Path
_here = Path.cwd()
for _c in [_here, *_here.parents]:
    if (_c / 'extension' / 'artifacts').exists():
        os.chdir(_c); break
sys.path.insert(0, str(Path.cwd()))
print('cwd:', os.getcwd())

cwd: /Users/tandon.utsav2/Desktop/Experiment_1


In [2]:
from extension.scripts.load_annotation_data import load_dataset
from extension.scripts import prompt_loader, extraction, scoring
from extension.scripts.model_specific import modal_kimi
from importlib import reload
modal_kimi = reload(modal_kimi)  # load script edits when this cell is rerun

assert modal_kimi.credentials_available(), (
    'Modal credentials not found; add MODAL-KEY and MODAL-SECRET to .env'
)
endpoint_model = modal_kimi.endpoint_info()
print('endpoint:', modal_kimi.endpoint_url())
print('served model:', endpoint_model['id'])
print('reasoning efforts:', [option['values'] for option in endpoint_model.get('reasoning_options', [])])

gold = load_dataset('extension/artifacts/annotation_dev_and_val_sets/validation_set.csv')
# The validation gold was carved from the MathDial train split and keeps its
# train indices. Its cache therefore lives under train so a later full-train
# run finds completed validation dialogues and skips them.
DIALOGUES = extraction.dialogues_from(gold, split='train')
UNITS = scoring.units_by_dialogue(gold)
print(f'{len(DIALOGUES)} dialogues, {len(gold)} units')

endpoint: https://utsavtandon96--ep-kimi-k3-server.us-west.modal.direct
served model: moonshotai/Kimi-K3
reasoning efforts: [['low', 'high', 'max']]
78 dialogues, 544 units


## Run configuration

This run explicitly uses `REASONING_EFFORT = "max"` and temperature `0.0`. The temperature is defined in `modal_kimi.py`; the reasoning effort is passed from this notebook. Alternative efforts (`"low"` and `"high"`) receive separate cache directories.

Two workers means at most two annotation requests are in flight. Responses are streamed so long max-reasoning generations send regular events instead of hitting the ten-minute buffered-response failure. Only pre-generation capacity responses (`429`, `503`, `504`) are retried automatically; a `502` is recorded once and the batch moves on.

In [3]:
TEST_PROMPTS = ['P1_full_codebook']
TEST_DIALOGUE_IDS = [323, 862, 822, 1717, 306, 434, 966, 736, 1051, 1119, 589, 992, 1063, 758, 275, 351, 1300, 448, 494]
MAX_WORKERS = 1
REASONING_EFFORT = 'max'
TEMPERATURE = 0.0
modal_kimi.TEMPERATURE = TEMPERATURE
TEST_MODEL = modal_kimi.cache_slug(REASONING_EFFORT)

In [4]:
_dialogues_by_id = {d['dialogue_id']: d for d in DIALOGUES}
TEST_DIALOGUES = [_dialogues_by_id[dialogue_id] for dialogue_id in TEST_DIALOGUE_IDS]
for _p in TEST_PROMPTS:
    assert _p in prompt_loader.list_prompts(), (
        f'unknown prompt {_p!r}; available: {prompt_loader.list_prompts()}'
    )
print(f'test model/cache: {TEST_MODEL} on dialogues '
      f"{[d['dialogue_id'] for d in TEST_DIALOGUES]} x prompts {TEST_PROMPTS}\n")

import json as _json
family_f1_cols = [f'f1_{family}' for family in scoring.FAMILIES]
test_rows = []
for prompt in TEST_PROMPTS:
    print(f'  {prompt} ({len(TEST_DIALOGUES)} dialogues, {MAX_WORKERS} workers):')
    modal_kimi.generate_annotations(
        prompt, TEST_DIALOGUES, reasoning_effort=REASONING_EFFORT,
        max_workers=MAX_WORKERS,
    )
    for dlg in TEST_DIALOGUES:
        rec = _json.load(open(extraction.cache_path(
            TEST_MODEL, prompt, dlg['dialogue_id'], dlg['split']
        )))
        attempt = rec.get('attempts', [{}])[-1] if rec.get('attempts') else {}
        tokens = (attempt.get('meta') or {}).get('tokens') or {}
        print(
            f"  {prompt:22s} {dlg['dialogue_id']}: "
            f"{'ok' if rec['valid'] else 'invalid':8s} "
            f"tokens {tokens.get('total', 0):,}  latency {rec['latency_s']:.1f}s"
        )
    s = scoring.score_config(
        gold, TEST_MODEL, prompt,
        [d['dialogue_id'] for d in TEST_DIALOGUES], n_boot=0, split='train'
    )
    test_rows.append(s)
    print(
        f"  -> validity {s['valid_rate']:.0%} | macro-F1(P) {s['macro_f1_P']:.3f} "
        f"| micro-F1(P) {s['micro_f1_P']:.3f} "
        f"| weighted-F1(P) {s['weighted_f1_P']:.3f} "
        f"| alpha {s['alpha']:.3f}"
    )
    print('     family F1(P): ' + ' | '.join(
        f"{family}={s[f'f1_{family}']:.3f}" for family in scoring.FAMILIES
    ) + '\n')

import pandas as pd
summary_cols = [
    'prompt', 'valid_rate', 'macro_f1_P', 'micro_f1_P',
    'weighted_f1_P', 'alpha',
    *family_f1_cols, 'latency_s',
]
summary = pd.DataFrame(test_rows)[summary_cols].round(3)
print(summary.to_string(index=False))

test model/cache: modal-endpoint/kimi-k3-max on dialogues [323, 862, 822, 1717, 306, 434, 966, 736, 1051, 1119, 589, 992, 1063, 758, 275, 351, 1300, 448, 494] x prompts ['P1_full_codebook']

  P1_full_codebook (19 dialogues, 1 workers):
  323: ok
  862: invalid
  822: ok
  1717: ok
  306: ok
  434: ok
  966: ok
  736: invalid
  1051: ok
  1119: ok
  589: ok
  992: ok
  1063: ok
  758: ok
  275: cached
  351: invalid
  1300: ok
  448: ok
  494: ok
  P1_full_codebook       323: ok       tokens 162,384  latency 589.5s
  P1_full_codebook       862: invalid  tokens 0  latency 3055.4s
  P1_full_codebook       822: ok       tokens 102,580  latency 458.4s
  P1_full_codebook       1717: ok       tokens 76,732  latency 261.9s
  P1_full_codebook       306: ok       tokens 72,742  latency 232.0s
  P1_full_codebook       434: ok       tokens 84,017  latency 383.0s
  P1_full_codebook       966: ok       tokens 97,617  latency 505.1s
  P1_full_codebook       736: invalid  tokens 0  latency 3185.5s
  

### Notes

Prompt files live in `extension/artifacts/annotation_prompts/`. Every response is cached with its raw output, reasoning trace, validation errors, usage, latency, endpoint metadata, streaming diagnostics, and request-attempt count. Partial content and reasoning are retained if a stream fails. Invalid records are retried on the next notebook execution; valid records are skipped.

The results are directly comparable on annotation metrics with the OpenRouter Kimi run because both use the same dialogues, prompt, validator, and scoring code. Dollar cost is not directly comparable from these cache files because Modal reports endpoint compute billing outside the completion response.